# AI-Augmented Technical Manuals - Streamlit Gemini RAG POC

This Colab notebook creates and runs a Streamlit proof of concept for A320 ATA 32-42 braking manuals. It uses local retrieval for technical context and Gemini for answer generation.


## Architecture

```text
Aircraft configuration + technician question
        |
        v
Local technical retrieval -> cited technical excerpts
        |
        v
Gemini answer generator -> cited answer with DMC/FIN/source family
```


In [ ]:
# Install dependencies
!pip install -q -U streamlit google-genai python-dotenv


In [ ]:
# Add your Gemini key. Recommended: Colab left sidebar -> Secrets -> GEMINI_API_KEY.
import os
from getpass import getpass
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY") or userdata.get("GOOGLE_API_KEY")
except Exception:
    key = None
if not key:
    key = getpass("Enter GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = key


In [ ]:
# Write app and embedded manuals to /content
from pathlib import Path
APP_DIR = Path("/content/airbus_brake_streamlit_poc")
(APP_DIR / "manuals").mkdir(parents=True, exist_ok=True)
APP_CODE = "import os\nimport re\nimport textwrap\nfrom pathlib import Path\nfrom typing import Optional\n\nimport streamlit as st\nfrom google import genai\n\n\nAPP_TITLE = \"AI-Augmented Technical Manuals\"\nDOC_DIR = Path(__file__).parent / \"manuals\"\nGEMINI_MODEL = os.getenv(\"GEMINI_MODEL\", \"gemini-2.5-flash\")\nCHUNK_SIZE = 1400\nCHUNK_OVERLAP = 180\nTOP_K = 6\nSTOP_WORDS = {\n    \"what\",\n    \"which\",\n    \"with\",\n    \"from\",\n    \"that\",\n    \"this\",\n    \"should\",\n    \"required\",\n    \"give\",\n    \"tell\",\n    \"show\",\n    \"and\",\n    \"the\",\n    \"for\",\n    \"are\",\n    \"is\",\n    \"in\",\n    \"on\",\n    \"to\",\n    \"of\",\n    \"a\",\n    \"an\",\n}\n\nDOC_FAMILY_BY_FILE = {\n    \"amm_brake_actuator_maintenance.md.txt\": \"AMM\",\n    \"ipc_brake_housing_components.md.txt\": \"IPC\",\n    \"wdm_brake_transducer_circuit.md.txt\": \"WDM\",\n    \"fim_brake_system_fault.md.txt\": \"FIM\",\n}\n\n\nst.set_page_config(\n    page_title=APP_TITLE,\n    page_icon=\"\",\n    layout=\"wide\",\n    initial_sidebar_state=\"collapsed\",\n)\n\nst.markdown(\n    \"\"\"\n    <style>\n    .block-container {padding-top: 1.5rem; max-width: 1180px;}\n    div[data-testid=\"stVerticalBlockBorderWrapper\"] {border-radius: 8px;}\n    .config-box {\n        border: 1px solid #2f3b52;\n        border-radius: 8px;\n        padding: 18px 18px 10px 18px;\n        background: #0f1724;\n        margin-bottom: 14px;\n    }\n    .config-title {\n        font-size: 1.05rem;\n        font-weight: 700;\n        margin-bottom: 14px;\n        color: #f7f9fc;\n    }\n    .source-card {\n        border: 1px solid #d9dee8;\n        border-radius: 8px;\n        padding: 12px;\n        margin-bottom: 10px;\n        background: #ffffff;\n    }\n    .small-muted {font-size: 0.85rem; color: #5f6b7a;}\n    </style>\n    \"\"\",\n    unsafe_allow_html=True,\n)\n\n\ndef get_secret(name: str) -> str:\n    if name in os.environ:\n        return os.environ[name]\n    try:\n        return st.secrets[name]\n    except Exception:\n        return \"\"\n\n\ndef make_client(api_key: str):\n    return genai.Client(api_key=api_key)\n\n\ndef clean_value(value: str) -> str:\n    value = re.sub(r\"\\s+\", \" \", value or \"\").strip()\n    return value.replace(\"\\u2013\", \"-\").replace(\"\\u2014\", \"-\")\n\n\ndef first_match(pattern: str, text: str, default: str = \"\") -> str:\n    match = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)\n    return clean_value(match.group(1)) if match else default\n\n\ndef extract_metadata(text: str, filename: str) -> dict:\n    family = DOC_FAMILY_BY_FILE.get(filename, \"MANUAL\")\n    return {\n        \"source\": family,\n        \"filename\": filename,\n        \"DMC\": first_match(r\"\\|\\s*\\*\\*DMC\\*\\*\\s*\\|\\s*([^|]+)\\|\", text, \"N/A\"),\n        \"FIN\": first_match(\n            r\"\\|\\s*\\*\\*(?:FIN|FIN \\(Primary\\)|Master Assembly FIN)\\*\\*\\s*\\|\\s*([^|]+)\\|\",\n            text,\n            \"N/A\",\n        ),\n        \"EFF\": first_match(r\"\\|\\s*\\*\\*(?:EFF|EFF \\(Document\\))\\*\\*\\s*\\|\\s*([^|]+)\\|\", text, \"N/A\"),\n        \"zone\": first_match(r\"\\|\\s*\\*\\*Zone\\*\\*\\s*\\|\\s*([^|]+)\\|\", text, \"N/A\"),\n        \"ATA\": first_match(r\"\\|\\s*\\*\\*ATA Chapter\\*\\*\\s*\\|\\s*([^|]+)\\|\", text, \"32-42\"),\n        \"fault_code\": first_match(r\"\\|\\s*\\*\\*Fault Code\\*\\*\\s*\\|\\s*`?([^`|]+)`?\\s*\\|\", text, \"\"),\n        \"connector\": first_match(r\"\\|\\s*\\*\\*(?:Connector Ref|Connector)\\*\\*\\s*\\|\\s*([^|]+)\\|\", text, \"\"),\n    }\n\n\ndef load_manuals() -> list[dict]:\n    manuals = []\n    for path in sorted(DOC_DIR.glob(\"*.txt\")):\n        text = path.read_text(encoding=\"utf-8\")\n        manuals.append(\n            {\n                \"filename\": path.name,\n                \"content\": text,\n                \"metadata\": extract_metadata(text, path.name),\n            }\n        )\n    return manuals\n\n\ndef chunk_text(text: str, metadata: dict) -> list[dict]:\n    parts = re.split(r\"(?=\\n### |\\n## )\", text)\n    chunks = []\n    current = \"\"\n    for part in parts:\n        part = part.strip()\n        if not part:\n            continue\n        if len(current) + len(part) <= CHUNK_SIZE:\n            current = f\"{current}\\n\\n{part}\".strip()\n        else:\n            if current:\n                chunks.append(current)\n            current = part\n            while len(current) > CHUNK_SIZE:\n                chunks.append(current[:CHUNK_SIZE])\n                current = current[CHUNK_SIZE - CHUNK_OVERLAP :]\n    if current:\n        chunks.append(current)\n\n    enriched = []\n    for index, chunk in enumerate(chunks):\n        row_eff = first_match(r\"\\*\\*EFF:\\s*([^*]+)\\*\\*\", chunk, metadata.get(\"EFF\", \"N/A\"))\n        enriched.append(\n            {\n                \"text\": chunk,\n                \"metadata\": {\n                    **metadata,\n                    \"row_eff\": row_eff,\n                    \"chunk_index\": index,\n                    \"chunk_id\": f\"{metadata['source']}-{index:03d}\",\n                },\n            }\n        )\n    return enriched\n\n\ndef build_chunks() -> list[dict]:\n    chunks = []\n    for manual in load_manuals():\n        chunks.extend(chunk_text(manual[\"content\"], manual[\"metadata\"]))\n    return chunks\n\n\ndef tokenize(text: str) -> list[str]:\n    tokens = re.findall(r\"[a-z0-9][a-z0-9-]{1,}\", text.lower())\n    return [token for token in tokens if token not in STOP_WORDS]\n\n\ndef prepare_search_text(text: str, metadata: dict) -> str:\n    return \" \".join(\n        [\n            text,\n            metadata.get(\"source\", \"\"),\n            metadata.get(\"DMC\", \"\"),\n            metadata.get(\"FIN\", \"\"),\n            metadata.get(\"EFF\", \"\"),\n            metadata.get(\"row_eff\", \"\"),\n            metadata.get(\"fault_code\", \"\"),\n            metadata.get(\"connector\", \"\"),\n        ]\n    ).lower()\n\n\ndef exact_phrase_score(question: str, search_text: str) -> float:\n    score = 0.0\n    quoted_terms = re.findall(r\"`([^`]+)`|\\\\b([A-Z]{2,}-[A-Z0-9-]+|[A-Z0-9]+-[A-Z0-9-]+)\\\\b\", question)\n    for groups in quoted_terms:\n        term = next((group for group in groups if group), \"\")\n        if term and term.lower() in search_text:\n            score += 3.0\n    return score\n\n\ndef lexical_score(question: str, item: dict) -> float:\n    search_text = item[\"search_text\"]\n    tokens = tokenize(question)\n    if not tokens:\n        return 0.0\n\n    score = exact_phrase_score(question, search_text)\n    for token in tokens:\n        count = search_text.count(token)\n        if not count:\n            continue\n        if any(char.isdigit() for char in token) or \"-\" in token:\n            score += min(count, 4) * 1.25\n        else:\n            score += min(count, 4) * 0.55\n\n    compact_question = \" \".join(tokens)\n    for phrase_len in (4, 3, 2):\n        words = compact_question.split()\n        for start in range(0, max(len(words) - phrase_len + 1, 0)):\n            phrase = \" \".join(words[start : start + phrase_len])\n            if phrase and phrase in search_text:\n                score += phrase_len * 0.75\n    return score\n\n\ndef reset_index():\n    st.session_state.pop(\"knowledge_index\", None)\n\n\ndef build_index(api_key: Optional[str] = None) -> int:\n    chunks = build_chunks()\n    st.session_state[\"knowledge_index\"] = [\n        {\n            \"text\": chunk[\"text\"],\n            \"metadata\": chunk[\"metadata\"],\n            \"search_text\": prepare_search_text(chunk[\"text\"], chunk[\"metadata\"]),\n        }\n        for chunk in chunks\n    ]\n    return len(chunks)\n\n\ndef collection_ready() -> bool:\n    return bool(st.session_state.get(\"knowledge_index\"))\n\n\ndef retrieve(api_key: str, question: str, families: list[str], top_k: int = TOP_K) -> list[dict]:\n    rows = []\n    for item in st.session_state.get(\"knowledge_index\", []):\n        if families and item[\"metadata\"].get(\"source\") not in families:\n            continue\n        score = lexical_score(question, item)\n        rows.append({\"text\": item[\"text\"], \"metadata\": item[\"metadata\"], \"score\": score})\n    return sorted(rows, key=lambda row: row[\"score\"], reverse=True)[:top_k]\n\n\ndef build_context(retrieved: list[dict]) -> str:\n    blocks = []\n    for index, item in enumerate(retrieved, 1):\n        meta = item[\"metadata\"]\n        blocks.append(\n            textwrap.dedent(\n                f\"\"\"\n                [SOURCE {index}]\n                Family: {meta.get('source')}\n                DMC: {meta.get('DMC')}\n                FIN: {meta.get('FIN')}\n                Effectivity: {meta.get('row_eff') or meta.get('EFF')}\n                Zone: {meta.get('zone')}\n                Filename: {meta.get('filename')}\n                Text:\n                {item['text']}\n                \"\"\"\n            ).strip()\n        )\n    return \"\\n\\n---\\n\\n\".join(blocks)\n\n\ndef answer_with_gemini(api_key: str, question: str, config: dict, retrieved: list[dict]) -> str:\n    client = make_client(api_key)\n    context = build_context(retrieved)\n    prompt = f\"\"\"\nYou are an expert Airbus A320 maintenance engineer supporting a proof of concept.\nUse ONLY the retrieved context. Do not invent values, part numbers, limits, authority, or steps.\nIf the answer is absent from the context, say that clearly.\n\nAircraft configuration:\n- Program family: {config['program_family']}\n- ATA chapter: {config['ata_chapter']}\n- Document families selected: {', '.join(config['families'])}\n- Effectivity: {config['effectivity']}\n\nRequired answer format:\n1. Direct answer\n2. Safety / caution items, if any\n3. Procedure or diagnostic steps, if relevant\n4. Source citations with DMC, FIN, document family, and filename\n\nRetrieved context:\n{context}\n\nTechnician question:\n{question}\n\"\"\"\n    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)\n    return response.text or \"No response text returned from Gemini.\"\n\n\ndef render_config_panel():\n    st.markdown('<div class=\"config-box\"><div class=\"config-title\">Aircraft Configuration</div>', unsafe_allow_html=True)\n    c1, c2, c3, c4 = st.columns([1.2, 1.6, 1.8, 1.2])\n    with c1:\n        program_family = st.selectbox(\"Program Family\", [\"A320\"], index=0)\n    with c2:\n        ata_chapter = st.selectbox(\"ATA Chapter\", [\"32-42 Braking System\"], index=0)\n    with c3:\n        st.caption(\"Document Family\")\n        f1, f2, f3, f4 = st.columns(4)\n        amm = f1.checkbox(\"AMM\", value=True)\n        ipc = f2.checkbox(\"IPC\", value=True)\n        wdm = f3.checkbox(\"WDM\", value=True)\n        fim = f4.checkbox(\"FIM\", value=True)\n    with c4:\n        effectivity = st.selectbox(\"Effectivity\", [\"MSN 001-200\", \"MSN 201-500\", \"ALL\"], index=0)\n    st.markdown(\"</div>\", unsafe_allow_html=True)\n    families = [name for name, enabled in [(\"AMM\", amm), (\"IPC\", ipc), (\"WDM\", wdm), (\"FIM\", fim)] if enabled]\n    return {\n        \"program_family\": program_family,\n        \"ata_chapter\": ata_chapter,\n        \"families\": families,\n        \"effectivity\": effectivity,\n    }\n\n\ndef render_architecture():\n    st.subheader(\"POC Architecture\")\n    st.code(\n        \"\"\"\nAircraft config + technician question\n        |\n        v\nTechnical publication access layer\n  - Program: A320\n  - ATA: 32-42 Braking System\n  - Approved document families\n  - Effectivity: MSN range\n        |\n        v\nLocal retrieval and effectivity filtering\n        |\n        v\nRelevant controlled-publication excerpts\n        |\n        v\nGemini answer generator\n        |\n        v\nGrounded answer + warnings + DMC/FIN citations\n        \"\"\".strip(),\n        language=\"text\",\n    )\n    st.markdown(\n        \"\"\"\nThis follows the deck idea of a smart knowledge layer over isolated technical manuals.\nThe visible experience abstracts away the underlying source count and presents the system as a controlled technical-publication layer.\n\nRecommended production upgrades:\n- Replace local markdown files with an approved technical-publication connector.\n- Persist retrieval indexes to controlled storage and version them by manual revision.\n- Add authentication, audit logs, and feedback capture.\n- Keep human-in-the-loop approval for any maintenance release decision.\n        \"\"\"\n    )\n\n\ndef render_sources(retrieved: list[dict]):\n    for index, item in enumerate(retrieved, 1):\n        meta = item[\"metadata\"]\n        st.markdown(\n            f\"\"\"\n            <div class=\"source-card\">\n            <b>Reference {index}: {meta.get('DMC')}</b><br/>\n            <span class=\"small-muted\">FIN: {meta.get('FIN')} | Effectivity: {meta.get('row_eff') or meta.get('EFF')}</span>\n            </div>\n            \"\"\",\n            unsafe_allow_html=True,\n        )\n        with st.expander(\"Show cited excerpt\"):\n            st.text(item[\"text\"][:1800])\n\n\nst.title(APP_TITLE)\nst.caption(\"Gemini-powered RAG POC for A320 ATA 32-42 braking manuals\")\n\napi_key_default = get_secret(\"GEMINI_API_KEY\") or get_secret(\"GOOGLE_API_KEY\")\nwith st.sidebar:\n    st.header(\"Runtime\")\n    api_key = st.text_input(\"Gemini API key\", value=api_key_default, type=\"password\")\n    st.caption(\"Gemini is used only for answer generation. Retrieval runs locally for demo stability.\")\n    reset = st.button(\"Reset knowledge layer\")\n    if reset:\n        reset_index()\n        st.success(\"Knowledge layer reset.\")\n\nconfig = render_config_panel()\n\ntab_ask, tab_arch = st.tabs([\"Ask Manuals\", \"Architecture\"])\n\nwith tab_ask:\n    if not api_key:\n        st.warning(\"Enter a Gemini API key in the sidebar before asking questions.\")\n\n    c1, c2 = st.columns([1, 1])\n    with c1:\n        if st.button(\"Initialize knowledge layer\", type=\"primary\"):\n            with st.spinner(\"Preparing the technical-publication knowledge layer...\"):\n                build_index()\n            st.success(\"Knowledge layer is ready.\")\n    with c2:\n        st.caption(\"First run prepares the local demonstration knowledge layer. Retrieval runs locally for stable demos.\")\n\n    examples = [\n        \"What torque is required for the structural retention bolts, and what sequence should I follow?\",\n        \"CMS-FAULT-32-42-E12 is shown. What is the first isolation step and possible causes?\",\n        \"For connector CN-LG32 Pin C, give the signal role, wire identifier, and termination.\",\n        \"Which actuator part number applies to MSN 001-200 and what O-ring kit is required?\",\n        \"What pressure must be confirmed before loosening any brake line union?\",\n    ]\n    question = st.text_area(\"Technician question\", value=examples[0], height=100)\n    top_k = st.slider(\"Evidence depth\", min_value=3, max_value=10, value=TOP_K)\n\n    if st.button(\"Ask Gemini\", disabled=not api_key or not config[\"families\"]):\n        if not collection_ready():\n            with st.spinner(\"Preparing the knowledge layer...\"):\n                build_index()\n        with st.spinner(\"Retrieving approved technical context...\"):\n            retrieved = retrieve(api_key, question, config[\"families\"], top_k=top_k)\n        with st.spinner(\"Generating grounded answer with Gemini...\"):\n            answer = answer_with_gemini(api_key, question, config, retrieved)\n        st.subheader(\"Answer\")\n        st.markdown(answer)\n        st.subheader(\"Traceability\")\n        render_sources(retrieved)\n\nwith tab_arch:\n    render_architecture()\n"
(APP_DIR / "app.py").write_text(APP_CODE, encoding="utf-8")
REQUIREMENTS = "streamlit==1.41.1\ngoogle-genai==1.0.0\npython-dotenv==1.0.1\n"
(APP_DIR / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
MANUALS = {"amm_brake_actuator_maintenance.md.txt": "# AIRCRAFT MAINTENANCE MANUAL (AMM)\n## Task 32-42-05-300-002: Removal and Installation \u2014 Main Brake Actuator Module\n\n---\n\n### DOCUMENT IDENTIFICATION\n\n| Metadata Key       | Value                                    |\n|--------------------|------------------------------------------|\n| **DMC**            | DMC-A320-A-32-42-05-00A-520A-A           |\n| **Task Number**    | 32-42-05-300-002                         |\n| **FIN**            | 10MG-A \u2014 Main Brake Actuator Module      |\n| **EFF**            | MSN 001\u2013200                              |\n| **Zone**           | 710 \u2014 Left Main Landing Gear Compartment |\n| **Access Panel**   | 711AR                                    |\n| **ATA Chapter**    | 32-42 \u2014 Normal Braking \u2014 Actuators       |\n| **Task Class**     | On-Aircraft Maintenance                  |\n| **Task Type**      | Removal / Installation                   |\n| **Authoring Std**  | S1000D Issue 4.2                         |\n| **Issue Date**     | 2024-11-15                               |\n| **Revision**       | C                                        |\n\n---\n\n### 1. REFERENCES\n\n| Reference Key                         | Title                                                        |\n|---------------------------------------|--------------------------------------------------------------|\n| `AMM Task 29-10-00-910-001`           | Hydraulic System \u2014 Blue System Depressurization              |\n| `AMM Task 29-10-00-300-001`           | Hydraulic System \u2014 Blue System Pressurization & Leak Check   |\n| `AMM Task 32-42-05-400-001`           | Brake Actuator Transducer \u2014 Bench Test Procedure             |\n| `AMM Task 20-93-00-400-001`           | Bonding and Earthing \u2014 Connector Backshell Procedure         |\n| `WDM DMC-A320-A-32-42-11-00A-520A-A` | Braking System \u2014 Wiring Diagram, Lines A/B Transducer Circuit|\n| `IPC Figure 32-42-05, Item 040`       | Main Brake Actuator Module \u2014 Part Number Reference           |\n| `AMM Task 12-32-00-680-001`           | Hydraulic Fluid Handling and Contamination Control           |\n\n---\n\n### 2. SAFETY REQUIREMENTS\n\n> \u26a0\ufe0f **WARNING:** The Blue Hydraulic System operates at a nominal pressure of\n> 3,000 PSI (206 bar). Residual trapped pressure in local brake lines may reach\n> 1,500 PSI (103 bar) even after system-level depressurization. Contact with\n> pressurized hydraulic fluid at this level constitutes a HIGH-PRESSURE INJECTION\n> HAZARD capable of penetrating skin and causing severe tissue necrosis, systemic\n> toxicity, or death. Do not loosen any hydraulic union, coupling, or bleed fitting\n> until system depressurization is confirmed at local test point TP-32A (pressure\n> \u2264 50 PSI / 3.4 bar as indicated on GSE-HYD-4200 gauge). Always direct drain\n> flow away from personnel. Wear face shield (EN166 rated), chemical-resistant\n> gloves (EN374 rated), and hydraulic-fluid-resistant coveralls before\n> disconnecting any fluid-carrying line.\n\n> \ud83d\uded1 **CAUTION:** The Main Brake Actuator Module (FIN: 10MG-A) contains internal\n> LVDT position transducers and pressure-sensing microcircuitry sensitive to\n> Electrostatic Discharge (ESD). Discharge to components at or above 100V may\n> cause latent or immediate failure of transducer signal processing circuits\n> without visible external damage. Before handling the actuator assembly or\n> its electrical connectors, don a wrist strap bonded to airframe earth\n> (resistance verified < 1 M\u03a9 per AMM Std. 20-93-00). Store removed unit\n> immediately in approved ESD-protective packaging (pink polyethylene\n> antistatic bag minimum; conductive hard-shell case preferred for transit).\n> Do not place unit on unprotected metal surfaces or synthetic-material\n> workbenches.\n\n> \u2139\ufe0f **NOTE:** This task requires a minimum crew of two (2) certified\n> maintenance personnel. One technician executes the procedure; a second\n> acts as safety observer and torque-witness. Both personnel must hold\n> ATA Chapter 32 task authorization on type. Estimated task duration:\n> 4.5 hours including bleed and leak-check. Schedule hydraulic system\n> availability accordingly.\n\n---\n\n### 3. PERSONNEL AND CERTIFICATION\n\n| Requirement          | Specification                              |\n|----------------------|--------------------------------------------|\n| Crew Minimum         | 2 persons                                  |\n| Authorization        | ATA 32 On-Aircraft Task Authorization      |\n| Medical Fitness      | Current \u2014 no impairment to fine motor task |\n| PPE Minimum Standard | Face shield EN166, Gloves EN374, Coveralls |\n\n---\n\n### 4. EXPENDABLE MATERIALS\n\n| Item | Description                          | Specification / P/N          | Qty     |\n|------|--------------------------------------|------------------------------|---------|\n| E01  | Hydraulic Fluid                      | Skydrol LD-4 or Hyjet IV-A   | 1.5 L   |\n| E02  | Blank Cap Set (hydraulic ports)      | ABS-CAP-32 (assorted)        | 1 set   |\n| E03  | O-Ring \u2014 Union Coupling (Primary)    | MS29513-016 (Viton)          | 2 each  |\n| E04  | O-Ring \u2014 Union Coupling (Return)     | MS29513-012 (Viton)          | 2 each  |\n| E05  | Locking Wire \u2014 Structural Bolts      | 0.032 in (0.8 mm) MS20995C32 | 300 mm  |\n| E06  | Lint-Free Wipe                       | AMM Std. Consumable C-020    | 10 each |\n| E07  | Antiseize Compound                   | Never-Seez NSBT-8 or equiv.  | As req. |\n| E08  | Bonding Lead \u2014 Temporary Earthing    | GSE-BOND-001                 | 1 each  |\n| E09  | Drain Tray (min. 2 L capacity)       | GSE-TRAY-002 or equiv.       | 1 each  |\n| E10  | ESD Protective Packaging             | Antistatic bag, min. 4 mil   | 1 each  |\n\n---\n\n### 5. TOOLING REQUIREMENTS\n\n| Tool ID | Description                                      | Part Number / Standard      |\n|---------|--------------------------------------------------|-----------------------------|\n| T01     | Flange Torque Spanner                            | ABS-9023                    |\n| T02     | Hydraulic Line Bleed Kit                         | Airbus-LK-32                |\n| T03     | Hydraulic Pressure Test Set (0\u20135,000 PSI)        | GSE-HYD-4200 or equiv.      |\n| T04     | Digital Torque Wrench (10\u2013100 ft-lbs / 14\u2013135 N\u00b7m) | Norbar Pro 100 or equiv.  |\n| T05     | Digital Torque Wrench (20\u2013200 in-lbs / 2.3\u201322.6 N\u00b7m) | Norbar Slipper or equiv. |\n| T06     | ESD Wrist Strap and Bonding Lead                 | Per AMM Std. 20-93-00       |\n| T07     | Face Shield (ballistic-rated)                    | EN166 certified             |\n| T08", "fim_brake_system_fault.md.txt": "# FAULT ISOLATION MANUAL (FIM/TSM)\n## TROUBLESHOOTING PROCEDURE \u2014 BRAKE SYSTEM PRESSURE FAULT\n\n---\n\n### DOCUMENT IDENTIFICATION\n\n| Metadata Key          | Value                                      |\n|-----------------------|--------------------------------------------|\n| **DMC**               | DMC-A320-A-32-42-00-00A-520A-A             |\n| **FIN**               | 10MG \u2014 Main Brake Actuator Assembly        |\n| **EFF**               | MSN 001\u2013200                                |\n| **Zone**              | 710 \u2014 Left Main Landing Gear Compartment   |\n| **Fault Code**        | CMS-FAULT-32-42-E12                        |\n| **ATA Chapter**       | 32-42 \u2014 Normal Braking \u2014 Actuators         |\n| **Issue Date**        | 2024-11-15                                 |\n| **Doc Class**         | TSM \u2014 Troubleshooting Manual               |\n| **Security Class**    | UNCLASSIFIED \u2014 Operator Distribution       |\n| **Authoring Std**     | S1000D Issue 4.2                           |\n\n---\n\n### 1. APPLICABILITY\n\n**EFF: MSN 001\u2013200**\n**Zone: 710 (Left Main Landing Gear Compartment)**\n**FIN: 10MG**\n\nThis procedure applies to aircraft with the Normal Braking System configured per\nDrawing D-32-42-000-00 Rev C and later. Do not apply to aircraft with SB 32-42-0031\nincorporated without concurrent incorporation of SB 32-42-0034.\n\n---\n\n### 2. ECAM ALERT REFERENCE\n\n| Field                    | Value                                                            |\n|--------------------------|------------------------------------------------------------------|\n| **ECAM Message**         | `BRAKE L/R SYS FAULT`                                           |\n| **Alert Class**          | Amber Caution                                                    |\n| **Fault Code**           | `CMS-FAULT-32-42-E12`                                           |\n| **CMS Trigger Condition**| Hydraulic pressure differential between Line A and Line B < 120 bar during active braking command |\n| **CMS Detection Logic**  | Pressure Transducer PT-32A vs. PT-32B mismatch exceeding \u00b115 bar threshold for > 2.0 sec continuous |\n| **Associated CFDS Class**| Class 1 \u2014 In-flight detectable, maintenance action required      |\n| **BITE Source**          | BSCU (Braking and Steering Control Unit) Ch. 1 / Ch. 2          |\n\n---\n\n### 3. SAFETY PRECAUTIONS\n\n> **WARNING:** Before performing any hydraulic system checks, ensure all hydraulic\n> system pressure is bled to zero. Residual pressure in Line A or Line B above\n> 50 bar constitutes a lethal injection hazard. Refer to AMM Task 29-10-00-910-001\n> (Hydraulic System Depressurization \u2014 Safety Procedure) before proceeding.\n\n> **CAUTION:** Do not actuate brakes during ground hydraulic power application\n> without installing wheel chocks. Failure to comply may result in uncommanded\n> aircraft movement.\n\n> **NOTE:** All CMS fault codes retrieved via the MCDU (CFDS menu path:\n> `MCDU \u2192 CFDS \u2192 AVNCS MENU \u2192 ATA 32 \u2192 BRAKING`) must be recorded prior to\n> any component removal. CMS data is volatile upon power interruption.\n\n---\n\n### 4. REFERENCED PUBLICATIONS\n\n| Reference Key                          | Document Title                                               |\n|----------------------------------------|--------------------------------------------------------------|\n| `AMM Task 32-42-05-300-002`            | Normal Brake Actuator \u2014 Functional Pressure Test             |\n| `WDM DMC-A320-A-32-42-11-00A-520A-A`  | Braking System \u2014 Wiring Diagram, Lines A/B Transducer Circuit|\n| `AMM Task 29-10-00-910-001`            | Hydraulic System \u2014 Depressurization Procedure                |\n| `AMM Task 32-42-00-810-001`            | Normal Braking System \u2014 BITE / CFDS Fault Retrieval          |\n| `IPC Figure 32-42-05, Item 070`        | Pressure Transducer PT-32A/B \u2014 Part Number Reference         |\n\n---\n\n### 5. TOOLS AND EQUIPMENT REQUIRED\n\n| Item | Description                            | Part Number / Standard  |\n|------|----------------------------------------|-------------------------|\n| T01  | Hydraulic Pressure Test Set            | GSE-HYD-4200 or equiv.  |\n| T02  | Digital Multimeter (CAT III, 600V)     | Per AMM Std. 70-00-00   |\n| T03  | CMS / CFDS MCDU Interface Cable        | GSE-MCDU-001            |\n| T04  | Bonding/Earthing Lead (2m min)         | Per AMM Std. 20-93-00   |\n\n---\n\n### 6. FAULT ISOLATION TREE\n\n**Fault Code: CMS-FAULT-32-42-E12**\n**Trigger:** Transducer pressure mismatch \u2014 Line A vs. Line B < 120 bar during braking command.\n\n| Step | Diagnostic Action                                                                                                                                                                                                                  | Finding / Result                                                                                   | Next Step / Resolution                                                                                                                 |\n|------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------|\n| 1    | **Retrieve CFDS Fault History.** Via MCDU CFDS menu, access ATA 32 fault log. Record all instances of `CMS-FAULT-32-42-E12`. Note flight phase at fault trigger (ground / airborne) and number of occurrences (last 10 flights). | **Finding A:** Single occurrence, airborne, no repeat. | Proceed to **Step 2** to isolate intermittent transducer fault. |\n|      |                                                                                                                                                                                                                                    | **Finding B:** Repeat occurrences (\u2265 3), any phase.                                                | Proceed directly to **Step 3**. High confidence hardware fault.                                                                        |\n|      |                                                                                                                                                                                                                                    | **Finding C:** Fault logged only on ground with no hydraulic power applied.                         | Proceed to **Step 4**. Suspect BSCU internal fault or ground power anomaly.                                                            |\n| 2    | **Perform Transducer Differential Pressure Test.** Apply hydraulic ground power (Line A and Line B simultaneously) per `AMM Task 32-42-05-300-002`. Connect pressure test set T01 at test points TP-32A and TP-32B (Zone 710, Fwd bulkhead, FS 2750). Command braking via hydraulic test rig. Record PT-32A and PT-32B output pressures at 50%, 75%, and 100% brake command. | **Finding A:** Both transducers read within \u00b110 bar of each other across all command levels. Pressure \u2265 140 bar at 100% command. | No hardware fault confirmed. Likely CAS transient. Clear fault in CFDS. Return aircraft to service. Document findings per local QA procedure. |\n|      |                                                                                                                                                                                                                                    | **Finding B:** PT-32A or PT-32B reading deviates > \u00b115 bar from counterpart at any command level. Identify which transducer is out-of-tolerance. | Proceed to **Step 3** with identified suspect transducer.                                                                              |\n|      |                                                                                                                                                                                                                                    | **Finding C:** Both transducers read < 120 bar at 100% brake command.                              | Suspect upstream hydraulic restriction or low system charge. Refer to `AMM Task 29-10-00-300-001` (Hydraulic System Servicing). Do not proceed on brake system until system pressure verified serviceable. |\n| 3    | **Perform Transducer Wiring and Continuity Check.** Depressurize hydraulic system per `AMM Task 29-10-00-910-001`. Disconnect electrical connectors CN-32A and CN-32B from suspect transducer (Zone 710, reference `WDM DMC-A320-A-32-42-11-00A-520A-A`, Sheet 3). Using DMM (T02), measure: (a) Pin 1\u20132 supply voltage (28VDC nominal), (b) Pin 3\u20134 signal output resistance (expected: 1.2 k\u03a9\u20132.4 k\u03a9), (c) Pin 5 shield/ground continuity to airframe (< 1 \u03a9). | **Finding A:** Supply voltage within tolerance. Signal resistance within range. Ground continuity confirmed. | Transducer electrically serviceable. Suspect BSCU channel processing fault. Proceed to **Step 4**.                                    |\n|      |                                                                                                                                                                                                                                    | **Finding B:** Supply voltage absent or < 24 VDC.                                                  | Trace wiring upstream to BSCU output per `WDM DMC-A320-A-32-42-11-00A-520A-A`, Sheet 1. Repair open circuit or replace wiring segment per `AMM Task 20-20-00-400-001`. Retest per Step 2. |\n|      |                                                                                                                                                                                                                                    | **Finding C:** Signal resistance out of range (< 1.2 k\u03a9 or > 2.4 k\u03a9), or open circuit.            | **Replace** suspect pressure transducer (PT-32A or PT-32B) per `AMM Task 32-42-05-400-001`. Reference `IPC Figure 32-42-05, Item 070` for P/N. Retest per Step 2. |\n|      |                                                                                                                                                                                                                                    | **Finding D:** Ground continuity > 1 \u03a9 (high resistance or open).                                  | Clean and re-bond connector backshell to airframe structure per `AMM Task 20-93-00-400-001`. Retest continuity before proceeding.      |\n| 4    | **BSCU Channel Isolation.** With hydraulic system depressurized and wiring confirmed serviceable, perform BSCU Built-In Test via CFDS: `MCDU \u2192 CFDS \u2192 AVNCS MENU \u2192 ATA 32 \u2192 BSCU \u2192 GND TEST \u2192 CHAN SEL`. Run Channel 1 and Channel 2 tests independently. Record PASS/FAIL status for each channel. | **Finding A:** Both channels PASS. No BSCU internal fault detected.                                | Fault non-reproducible at component level. Clear fault code in CFDS. Carry forward in Tech Log per operator MEL/DDG. Monitor for recurrence over next 5 flights. |\n|      |                                                                                                                                                                                                                                    | **Finding B:** Channel 1 FAIL, Channel 2 PASS (or vice versa).                                    | **Replace BSCU** per `AMM Task 32-43-00-400-001`. Following replacement, perform full BSCU ground test per `AMM Task 32-43-00-710-001`. Retest brake system per Step 2. |\n|      |                                                                                                                                                                                                                                    | **Finding C:** Both channels FAIL.                                                                 | **Replace BSCU** per `AMM Task 32-43-00-400-001`. Verify aircraft-side wiring and connector integrity before installing new unit. Perform full BSCU ground test per `AMM Task 32-43-00-710-001`. If fault persists post-replacement, escalate to Airbus Tech Support (AOG Desk) with full CFDS dump. |\n\n---\n\n### 7. POST-ISOLATION ACTIONS\n\n1. Clear all active faults from CFDS following confirmation of corrective action.\n2. Perform a complete brake system operational check per `AMM Task 32-42-05-300-002`.\n3. Verify no residual ECAM messages present on both ECAM DUs prior to releasing aircraft.\n4. Complete all entries in the Aircraft Technical Log (ATL) and Maintenance Record per operator procedures.\n5. If any component was replaced (transducer or BSCU), update Component History Record (CHR) per operator CMM traceability requirements.\n\n> **NOTE:** If fault `CMS-FAULT-32-42-E12` recurs within 5 flight cycles following\n> any corrective action defined in this procedure, escalate to Tier 2 troubleshooting\n> and contact Airbus Customer Services \u2014 Technical AOG Desk. Provide full CFDS data\n> export and component history records for engineering review.\n\n---\n\n### 8. REVISION RECORD\n\n| Rev | Date       | Reason for Change                          | Author     |\n|-----|------------|--------------------------------------------|------------|\n| A   | 2023-04-10 | Initial issue                              | Tech Pubs  |\n| B   | 2024-01-22 | Step 3 wiring tolerance updated per SB 32-42-0031 | Tech Pubs  |\n| C   | 2024-11-15 | Added BSCU dual-channel isolation (Step 4) | Tech Pubs  |\n\n---\n\n*End of Data Module \u2014 DMC-A320-A-32-42-00-00A-520A-A*", "ipc_brake_housing_components.md.txt": "# ILLUSTRATED PARTS CATALOG (IPC)\n## ATA Chapter 32-42-01: Main Brake Housing Assembly Components\n\n---\n\n### DOCUMENT IDENTIFICATION\n\n| Metadata Key              | Value                                          |\n|---------------------------|------------------------------------------------|\n| **DMC**                   | DMC-A320-A-32-42-01-00A-520A-A                 |\n| **Document Type**         | IPC \u2014 Illustrated Parts Catalog                |\n| **ATA Chapter**           | 32-42-01 \u2014 Main Brake Housing Assembly         |\n| **Master Assembly FIN**   | 10MG \u2014 Main Brake Actuator Assembly Group      |\n| **EFF (Document)**        | ALL (row-level effectivity governs per item)   |\n| **Zone**                  | 710 \u2014 Left Main Landing Gear Compartment       |\n| **Authoring Std**         | S1000D Issue 4.2 / ATA iSpec 2200              |\n| **Issue Date**            | 2024-11-15                                     |\n| **Revision**              | C                                              |\n| **Security Class**        | UNCLASSIFIED \u2014 Operator and Vendor Distribution|\n\n---\n\n### REFERENCES\n\n| Reference Key                         | Document Title                                              |\n|---------------------------------------|-------------------------------------------------------------|\n| `AMM Task 32-42-05-300-002`           | Main Brake Actuator Module \u2014 Removal and Installation       |\n| `AMM Task 32-42-05-400-001`           | Brake Actuator Transducer \u2014 Bench Test Procedure            |\n| `WDM DMC-A320-A-32-42-11-00A-520A-A` | Braking System \u2014 Wiring Diagram, Lines A/B Transducer Circuit|\n| `TSM DMC-A320-A-32-42-00-00A-520A-A` | Braking System \u2014 Fault Isolation Manual                     |\n| `IPC Figure 32-42-01`                 | Main Brake Housing Assembly \u2014 Exploded View Illustration    |\n| `SB A320-32-1042`                     | Service Bulletin \u2014 Actuator Base Mod Transition MSN 001\u2013200 to 201\u2013500 |\n| `MIL-DTL-38999L`                      | Connector, Electrical, Circular, Miniature, Specification   |\n\n---\n\n### USAGE NOTES\n\n> \u2139\ufe0f **NOTE:** Each row in the parts matrix below contains a self-contained\n> Effectivity (EFF) string. This architecture ensures that LLM text-chunk\n> retrieval operations on individual rows return unambiguous part-to-airframe\n> compatibility data without dependency on document-level metadata context.\n> Always verify row-level EFF against the aircraft MSN before raising a\n> purchase order or work order.\n\n> \u2139\ufe0f **NOTE:** Items 1 and 1A are mutually exclusive effectivity variants of\n> the same installation position. Do not install P/N BA-900-3242A on MSN 201\n> and above. Do not install P/N BA-900-3242B on MSN 001\u2013200 without prior\n> incorporation of Service Bulletin `SB A320-32-1042`.\n\n---\n\n### PARTS MATRIX \u2014 FIN: 10MG \u2014 MAIN BRAKE HOUSING ASSEMBLY\n\n| Item Number | Part Number     | Nomenclature / Description                                                                   | Qty per Assy | Effectivity (EFF) / Tail Application                                                      |\n|-------------|-----------------|----------------------------------------------------------------------------------------------|:------------:|-------------------------------------------------------------------------------------------|\n| 1           | BA-900-3242A    | Main Brake Actuator Assembly \u2014 Base Production Modification, Standard-Cycle Configuration, Aluminum Housing, Nitrile Primary Seals, Operating Pressure 3,000 PSI (206 bar) | 1 | **EFF: MSN 001\u2013200** \u2014 Applicable to all aircraft from first delivery through MSN 200. Superseded by BA-900-3242B on MSN 201 and above per SB A320-32-1042. Not interchangeable with Item 1A without SB embodiment. |\n| 1A          | BA-900-3242B    | Main Brake Actuator Assembly \u2014 Upgraded High-Cycle Modification, Extended-Life Ceramic-Coated Bore, PTFE Primary Seals, Operating Pressure 3,000 PSI (206 bar), Fatigue Life +40% vs. BA-900-3242A | 1 | **EFF: MSN 201\u2013500** \u2014 Applicable to all aircraft from MSN 201 through MSN 500 inclusive. Retrofit to MSN 001\u2013200 is permissible only after compliance with SB A320-32-1042. Item 1 and Item 1A are position-exclusive; only one installed per aircraft. |\n| 2           | PT-402-AERO     | Hydraulic Pressure Transducer \u2014 Piezo-resistive Full Wheatstone Bridge, Range 0\u20133,000 PSI (0\u2013206 bar), Output 1.0 VDC\u20135.0 VDC Analog Linear, Excitation +28 VDC, MIL-PRF-39001 Class S rated, M18 \u00d7 1.5 port thread, Connector per MIL-DTL-38999 Series III 4-pin | 1 | **EFF: ALL** \u2014 Applicable to all MSN without restriction. COTS component; no service bulletin dependency. Interchangeable across FIN 10MG installations on MSN 001\u2013500. |\n| 3           | OR-3242-K       | O-Ring Replacement Overhaul Kit \u2014 Nitrile (NBR 70 Shore A), Skydrol LD-4 and Hyjet IV-A compatible, kit contains: (2) MS29513-016 Primary Union O-Rings, (2) MS29513-012 Return Union O-Rings, (1) MS29513-906 End-Cap Seal, (4) MS29513-008 Port Plug O-Rings; all items individually bagged and lot-traced; shelf life 10 years from cure date per AS1933 | 1 | **EFF: ALL** \u2014 Applicable to all MSN without restriction. Mandatory replacement at every actuator removal/installation event per AMM Task 32-42-05-300-002. Do not reuse removed O-rings regardless of apparent condition. |\n| 4           | B-0375-16A      | Structural Retention Bolt \u2014 Hex Head, Material: Alloy Steel ASTM A574, Surface Treatment: Zinc-Nickel Plate per ASTM B841, Grade 12.9, Size: 3/8 in \u00d7 2 in (M10 \u00d7 50 mm metric equivalent), Thread: 3/8-16 UNC-2A, Head Drive: Hex 9/16 in (14.3 mm) AF, Tensile Strength: 180,000 PSI (1,241 MPa) minimum, Installation Torque: 45 ft-lbs (61 N\u00b7m) per AMM Task 32-42-05-300-002 cross-torque sequence, Locking Wire required: MS20995C32 | 3 | **EFF: ALL** \u2014 Applicable to all MSN without restriction. Life-limited: maximum 5,000 installation cycles per IPC Figure 32-42-01. Inspect for thread damage and corrosion at each removal. Replace as a complete set of 3; do not mix lot numbers within one installation position. |\n| 5           | CN-LG32-P       | Plug Connector Sub-Assembly \u2014 Environmentally Sealed, Circular, 4-Pin, MIL-DTL-38999 Series III, Shell Size 11, Insert Arrangement 4-4, Key Position A, Bayonet Coupling, Shell Material Aluminum Alloy 6061-T6 Cadmium Plated Olive Drab, Contacts Gold-Plated Crimp Size 20 per MIL-C-39029/4, Environmental Harshness Zone 5 rated (-55\u00b0C to +200\u00b0C, IP67), 90\u00b0 angled backshell ABS-BKSHL-38999-11-90D included, mates with socket CN-LG32 per WDM DMC-A320-A-32-42-11-00A-520A-A | 1 | **EFF: ALL** \u2014 Applicable to all MSN without restriction. Replace as complete sub-assembly; do not replace individual contacts only unless contact damage is isolated and wire harness W-32-MG100-SERIES is verified undamaged. Inspect at each actuator removal per AMM Task 32-42-05-300-002 Step 7.2. |\n| 6           | BS-885-M32      | Machined Brake Stator Housing Module \u2014 Aluminum Alloy 7075-T73, Hard Anodize Finish MIL-A-8625 Type III Class 2, Bore Diameter 88.5 mm \u00b1 0.01 mm (3.484 in \u00b1 0.0004 in), Port Thread P1: M18 \u00d7 1.5 (Primary Pressure), Port Thread P2: M14 \u00d7 1.5 (Return), Dowel Pin Bore: 12.00 mm +0.00/\u22120.05 mm, Mounting Flange: 4 \u00d7 M12 threaded inserts, Stator Slot Count: 5, Heat Treatment per AMS 2770, Dimensional Inspection Report required at installation | 1 | **EFF: MSN 001\u2013200** \u2014 Applicable to all aircraft from MSN 001 through MSN 200 inclusive. Superseded on MSN 201 and above by BS-886-M32 (not within scope of this IPC revision). Bore wear limit: 12.15 mm maximum on dowel bores per SRM Chapter 32-42. Return unserviceable units to overhaul with full dimensional report. |\n\n---\n\n### PROCUREMENT METADATA BLOCK\n\n#### Classification and Source Coding\n\n| Item Number | Part Number  | Procurement Class | Source Category                  | CAGE Code | Vendor Name                        | Lead Time (Typical) | MOQ |\n|-------------|--------------|-------------------|----------------------------------|-----------|-------------------------------------|---------------------|-----|\n| 1           | BA-900-3242A | Class 1 \u2014 Proprietary | Airbus Direct Supply Only   | V77291    | Airbus Propulsion Systems           | 90\u2013120 calendar days | 1   |\n| 1A          | BA-900-3242B | Class 1 \u2014 Proprietary | Airbus Direct Supply Only   | V77291    | Airbus Propulsion Systems           | 90\u2013120 calendar days | 1   |\n| 2           | PT-402-AERO  | COTS              | Regional Warehouse Stock         | N/A       | Multi-source \u2014 Approved Vendor List | 5\u201310 business days  | 1   |\n| 3           | OR-3242-K    | COTS              | Regional Warehouse Stock         | N/A       | Multi-source \u2014 Approved Vendor List | 3\u20137 business days   | 1   |\n| 4           | B-0375-16A   | COTS              | Regional Warehouse Stock         | N/A       | Multi-source \u2014 Approved Vendor List | 3\u20137 business days   | 6   |\n| 5           | CN-LG32-P    | COTS              | Regional Warehouse Stock         | N/A       | Multi-source \u2014 Approved Vendor List | 5\u201310 business days  | 1   |\n| 6           | BS-885-M32   | Class 1 \u2014 Proprietary | Airbus Direct Supply Only   | V77291    | Airbus Propulsion Systems           | 120\u2013180 calendar days | 1  |\n\n---\n\n#### Procurement Class Definitions\n\n| Class                      | Definition                                                                                                   |\n|----------------------------|--------------------------------------------------------------------------------------------------------------|\n| **Class 1 \u2014 Proprietary**  | Part manufactured exclusively by or under direct license from Airbus Propulsion Systems (CAGE: V77291). Procurement through any channel other than the Airbus Spares Network (ASN) or an authorized Airbus Spare Parts Center (SPC) is prohibited. No approved alternate source exists. Requests must be raised via Airbus Material Order System (AMOS) with full MSN and FIN declaration. |\n| **COTS**                   | Commercial-Off-The-Shelf part procurable from any supplier on the Airbus Approved Vendor List (AVL) for ATA Chapter 32. Regional Warehouse Stock items are pre-positioned at operator line stations. Purchase Orders must reference the Airbus P/N exactly as listed in this IPC. Substitution of equivalent commercial P/Ns without Airbus Engineering Authorization (AEA) is prohibited. |\n\n---\n\n#### Stockroom and Handling Requirements\n\n| Item Number | Part Number  | Storage Class      | ESD Sensitivity | Shelf Life       | Handling Requirement                                              |\n|-------------|--------------|--------------------|-----------------|------------------|-------------------------------------------------------------------|\n| 1           | BA-900-3242A | Controlled \u2014 Bonded Store | Non-ESD    | Unlimited (inspect seals at 5-year intervals) | Store horizontally in original packaging. Protect port threads with plastic caps. Inspect O-ring pre-seals before installation. |\n| 1A          | BA-900-3242B | Controlled \u2014 Bonded Store | Non-ESD    | Unlimited (inspect seals at 5-year intervals) | Store horizontally in original packaging. Protect port threads with plastic caps. Inspect O-ring pre-seals before installation. |\n| 2           | PT-402-AERO  | Standard Avionics Store | ESD Sensitive \u2014 Class 1A | Unlimited (bench-test after 24 months) | Handle with ESD wrist strap. Store in antistatic bag. Do not expose connector pins to uncontrolled environment. |\n| 3           | OR-3242-K    | Elastomer Store \u2014 Temperature Controlled (+15\u00b0C to +25\u00b0C) | Non-ESD | 10 years from cure date (lot-trace label) | Do not store near ozone sources, UV light, or solvents. Verify cure date before issue. Return to stores if cure date is within 6 months of expiry. |\n| 4           | B-0375-16A   | Standard Hardware Store | Non-ESD    | Unlimited (inspect coating at issue) | Inspect zinc-nickel plate for corrosion before issue. Issue as matched lot set of 3. Do not issue individual bolts from mixed lots. |\n| 5           | CN-LG32-P    | Avionics Store      | Non-ESD (metallic shell) \u2014 ESD Sensitive (internal contact surfaces) | Unlimited | Inspect contact gold plating and keyway before issue. Store in sealed original packaging. |\n| 6           | BS-885-M32   | Controlled \u2014 Bonded Store | Non-ESD    | Unlimited (dimensional re-inspection after 10 years in storage) | Store with bore plugged and ports capped. Keep dimensional inspection certification with unit. Do not issue without current dimensional report on file. |\n\n---\n\n### REVISION RECORD\n\n| Rev | Date       | Reason for Change                                                          | Author          |\n|-----|------------|----------------------------------------------------------------------------|-----------------|\n| A   | 2022-06-01 | Initial issue \u2014 Items 1, 2, 3, 4, 5 baselined                             | Config. Mgmt.   |\n| B   | 2023-03-14 | Item 1A added per SB A320-32-1042; Item 6 added; CAGE code verified        | Config. Mgmt.   |\n| C   | 2024-11-15 | Stockroom handling table added; O-ring shelf life updated to 10 years; Item 4 MOQ corrected to 6 | Config. Mgmt.   |\n\n---\n\n*End of Data Module \u2014 DMC-A320-A-32-42-01-00A-520A-A*\n*IPC ATA Chapter 32-42-01 \u2014 Main Brake Housing Assembly Components \u2014 Revision C \u2014 2024-11-15*", "wdm_brake_transducer_circuit.md.txt": "```markdown\n# AIRCRAFT WIRING MANUAL (AWM/WDM)\n## Wiring Architecture and Pin-Out Routing Specification\n### Brake Pressure Transducer Signal Circuit \u2014 Lines A/B Interface\n\n---\n\n### DOCUMENT IDENTIFICATION\n\n| Metadata Key         | Value                                              |\n|----------------------|----------------------------------------------------|\n| **DMC**              | DMC-A320-A-32-42-11-00A-520A-A                     |\n| **Document Type**    | AWM/WDM \u2014 Wiring Diagram and Pin-Out Specification |\n| **ATA Chapter**      | 32-42 \u2014 Normal Braking \u2014 Actuators                 |\n| **FIN (Primary)**    | 12MG \u2014 Brake Pressure Transducer                   |\n| **FIN (Secondary)**  | 10MG \u2014 Actuator System Control Unit Interface      |\n| **EFF**              | ALL                                                |\n| **Zone**             | 710 \u2014 Left Main Landing Gear Strut, Lower Forward Segment |\n| **Connector Ref**    | CN-LG32                                            |\n| **Authoring Std**    | S1000D Issue 4.2 / ASD-STE100                      |\n| **Issue Date**       | 2024-11-15                                         |\n| **Revision**         | D                                                  |\n| **Security Class**   | UNCLASSIFIED \u2014 Operator Distribution               |\n\n---\n\n### 1. REFERENCES\n\n| Reference Key                          | Document Title                                                  |\n|----------------------------------------|-----------------------------------------------------------------|\n| `AMM Task 32-42-05-300-002`            | Main Brake Actuator Module \u2014 Removal and Installation           |\n| `AMM Task 32-42-05-400-001`            | Brake Actuator Transducer \u2014 Bench Test Procedure                |\n| `TSM DMC-A320-A-32-42-00-00A-520A-A`  | Braking System \u2014 Fault Isolation Manual                         |\n| `WDM DMC-A320-A-32-42-11-01A-520A-A`  | Braking System \u2014 Wiring Diagram, BSCU Internal Bus Routing      |\n| `WDM DMC-A320-A-32-41-11-00A-520A-A`  | Braking System \u2014 Anti-Skid Control Wiring Schematic             |\n| `AWM Chapter 20-54`                    | Shielded Cable Routing and Termination Standard                 |\n| `AWM Chapter 24-50`                    | Aircraft DC Power Distribution \u2014 Circuit Breaker Panel 10VU     |\n| `IPC Figure 32-42-11, Item 010`        | Connector CN-LG32 \u2014 Part Number and Insertion Tool Reference    |\n| `MIL-DTL-38999L`                       | Connector, Electrical, Circular, Miniature, Specification       |\n| `MIL-W-22759/16`                       | Wire, Electrical, PTFE Insulated, Specification                 |\n\n---\n\n### 2. SCOPE AND PURPOSE\n\nThis data module defines the complete wiring architecture, conductor routing,\nconnector specification, and pin-out assignment for the analog pressure telemetry\ncircuit originating at the Brake Pressure Transducer (FIN: 12MG) in Zone 710,\nrouted via harness W-32-MG100-SERIES to the Brake System Control Unit Channel 1\n(BSCU-1) installed at Main Avionics Rack 8VU, and return-grounded at Ground\nBlock GD-02 (Frame 24, Zone 120).\n\nThis document provides the authoritative pin-to-pin wiring source for:\n- Connector maintenance, continuity verification, and pin extraction/insertion.\n- Fault isolation of signal anomalies on the 1.0 V\u20135.0 V analog telemetry loop.\n- EMI shielding integrity verification at bulkhead termination points.\n- Replacement of harness sub-segments identified in AWM Chapter 20-54.\n\n---\n\n### 3. SYSTEM SCHEMATIC SUMMARY\n\n#### 3.1 Signal Chain Architecture\n\nThe brake pressure telemetry loop is a single-ended analog voltage signal\noperating in the range **1.0 VDC (0 bar) to 5.0 VDC (350 bar)**, generated\nby the piezo-resistive bridge of the Brake Pressure Transducer FIN 12MG\n(Zone 710, Left Main Landing Gear Strut Lower Forward Segment).\n\nThe signal chain is defined as follows:\n\n```\n  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n  \u2502  DC POWER ORIGIN                                                                \u2502\n  \u2502  Panel 10VU \u2014 Circuit Breaker CB-10MG                                          \u2502\n  \u2502  Rating: 1A / 28 VDC \u2014 Location: Row E, Position 14, Panel 10VU               \u2502\n  \u2502  (Main Avionics Rack Zone, Zone 120, Frame 20)                                 \u2502\n  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n                               \u2502 +28 VDC Supply\n                               \u2502 Wire: W-32-MG102-A22 (22 AWG, Red, PTFE)\n                               \u2502 Routing: Zone 120 \u2192 Zone 310 \u2192 Zone 710\n                               \u2502 Via: Bulkhead Grommet BG-32-F24 at Frame 24\n                               \u25bc\n  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n  \u2502  CONNECTOR CN-LG32 \u2014 PIN A                                                     \u2502\n  \u2502  MIL-DTL-38999 Series III \u2014 4-Pin Bayonet Socket                               \u2502\n  \u2502  Zone 710 \u2014 Left Main Landing Gear Strut, Lower Forward Segment                \u2502\n  \u2502  Environmental Harshness Zone 5 (High vibration, fluid exposure, temp: -55\u00b0C   \u2502\n  \u2502  to +200\u00b0C)                                                                    \u2502\n  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n                               \u2502 +28 VDC Excitation to Transducer Bridge\n                               \u25bc\n  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n  \u2502  BRAKE PRESSURE TRANSDUCER \u2014 FIN: 12MG                                         \u2502\n  \u2502  Location: Zone 710, Left Main Gear Strut, FS 2762, Fwd Face                   \u2502\n  \u2502  Type: Piezo-resistive full Wheatstone bridge                                  \u2502\n  \u2502  Range: 0\u2013350 bar / 0\u20135,076 PSI                                                \u2502\n  \u2502  Output Signal: 1.0 VDC (0 bar) to 5.0 VDC (350 bar) analog linear            \u2502\n  \u2502  Excitation: +28 VDC \u00b10.5 V                                                    \u2502\n  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n           \u2502 Analog Signal Output                      \u2502 DC Ground Return\n           \u2502 Wire: W-32-MG104-C24 (24 AWG, Blue)       \u2502 Wire: W-32-MG103-B22 (22 AWG, Black)\n           \u2502 Twisted Shielded Pair with Pin D           \u2502\n           \u25bc                                           \u25bc\n  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n  \u2502  CONNECTOR CN-LG32 \u2014 PIN C (Signal) and PIN B (Ground)                         \u2502\n  \u2502  Twisted Shielded Pair: W-32-MG104-C24 / W-32-MG103-B22                        \u2502\n  \u2502  Shield: W-32-MG105-D-SH \u2014 Outer Braid, grounded at Bulkhead Frame 24          \u2502\n  \u2502  Shield Drain Wire terminates at CN-LG32 Pin D                                 \u2502\n  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n                               \u2502 Signal and Ground routed via harness\n                               \u2502 W-32-MG100-SERIES\n                               \u2502 Routing: Zone 710 \u2192 Zone 310 \u2192 Zone 120\n                               \u2502 Via: Conduit C-32-LG-01, tied at 300 mm intervals\n                               \u25bc\n  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n  \u2502  BRAKE SYSTEM CONTROL UNIT CHANNEL 1 \u2014 BSCU-1                                  \u2502\n  \u2502  FIN: 10MG \u2014 Actuator System Control Unit Interface                             \u2502\n  \u2502  Location: Main Avionics Rack 8VU, Shelf 3, Position 2                         \u2502\n  \u2502  Zone: 120 (Main Equipment Centre, Frame 18\u201322)                                \u2502\n  \u2502  Signal Input Impedance: 100 k\u03a9 (analog voltage input channel AV-12)           \u2502\n  \u2502  Accepted Signal Range: 0.8 VDC \u2013 5.2 VDC (0.8 V and 5.2 V = out-of-range    \u2502\n  \u2502  flags; triggers CMS-FAULT-32-42-E12 if sustained > 2.0 sec)                  \u2502\n  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n                               \u2502\n                               \u2502 DC Ground Return Path\n                               \u25bc\n  \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n  \u2502  GROUND BLOCK GD-02                                                             \u2502\n  \u2502  Location: Frame 24, Zone 120, Fwd Bulkhead Lower Stringer                     \u2502\n  \u2502  Type: 12-lug tinned copper bus bar, M4 stud terminations                      \u2502\n  \u2502  Bonded to airframe primary structure at Frame 24 center web                   \u2502\n  \u2502  Ground resistance to airframe: < 2.5 m\u03a9 (verified per AWM Ch. 20-93)         \u2502\n  \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n```\n\n#### 3.2 EMI Shield Architecture\n\nThe twisted shielded pair carrying Pin C (signal) and Pin B (ground) is enclosed\nin an outer braid shield (Wire Identifier: W-32-MG105-D-SH). Shield termination\nfollows AWM Chapter 20-54 single-point ground discipline:\n\n- **Transducer end (Zone 710):** Shield is NOT grounded. Braid is floating,\n  terminated with insulating heat-shrink sleeve at the CN-LG32 backshell entry.\n- **Bulkhead end (Frame 24):** Shield braid drain wire terminates at CN-LG32\n  Pin D, which routes via harness to Ground Block GD-02 at Frame 24.\n- **Single-point grounding** eliminates ground loop interference on the\n  1.0 V\u20135.0 V low-level analog signal path.\n- **Shield coverage:** Minimum 90% optical coverage per MIL-W-22759 braid\n  specification. Verify coverage before harness installation using continuity\n  check from braid-to-Pin D: target resistance < 0.5 \u03a9 end-to-end.\n\n---\n\n### 4. CONNECTOR SPECIFICATION \u2014 CN-LG32\n\n| Field                        | Specification                                                  |\n|------------------------------|----------------------------------------------------------------|\n| **Connector Reference**      | CN-LG32                                                        |\n| **Connector Type**           | Circular, Miniature, Bayonet Coupling, Socket (Receptacle)     |\n| **Specification**            | MIL-DTL-38999 Series III                                       |\n| **Shell Size**               | Size 11 (Nominal shell diameter: 0.811 in / 20.6 mm)          |\n| **Insert Arrangement**       | 4-Pin (Insert Arrangement Code: 4-4)                           |\n| **Coupling Mechanism**       | Bayonet \u2014 3-start, anti-decoupling ratchet                     |\n| **Shell Material**           | Aluminum Alloy 6061-T6, Electroless Nickel over Cadmium Plate  |\n| **Shell Finish**             | Olive Drab Cadmium per MIL-DTL-38999L Table I                  |\n| **Insert Material**          | Glass-filled fluorosilicone (GFS), 200\u00b0C rated                 |\n| **Contact Type**             | Crimp socket, size 20, gold-plated per MIL-C-39029/4           |\n| **Contact Material**         | Copper alloy, gold flash over nickel underplate                |\n| **Mating Plug Reference**    | CN-LG32P (Plug, field-installable, matching P/N per IPC 32-42-11, Item 010) |\n| **Environmental Harshness**  | Zone 5 \u2014 High vibration (15g, 20\u20132000 Hz), fluid immersion exposure, thermal cycling -55\u00b0C to +200\u00b0C |\n| **IP Rating**                | IP67 \u2014 Dust-tight and immersion to 1 m depth per IEC 60529    |\n| **Keying**                   | Key Position A (0\u00b0 reference, factory-set, non-interchangeable with adjacent connectors CN-LG30, CN-LG31, CN-LG33) |\n| **Backshell Type**           | ABS-BKSHL-38999-11-90D \u2014 90\u00b0 angled, EMI/RFI conductive, strain-relief clamp |\n| **Mounting**                 | Panel-mount, jam-nut secured, Zone 710 bracket BKT-32-LG-04 (FS 2762) |\n| **Mating Torque**            | 40 in-lbs (4.5 N\u00b7m) \u2014 bayonet coupling nut, per IPC Figure 32-42-11 |\n| **Unmating Torque**          | 15 in-lbs (1.7 N\u00b7m) minimum pull-off \u2014 anti-decoupling ratchet |\n| **Approved P/N (Socket)**    | AMP / TE Connectivity P/N: D38999/26WA-4SN                    |\n| **Approved P/N (Plug)**      | AMP / TE Connectivity P/N: D38999/26FB-4PN                    |\n\n---\n\n### 5. CONNECTOR FACE VIEW \u2014 CN-LG32 PIN ARRANGEMENT\n\nThe following ASCII diagram represents the **mating face view** of the socket\nreceptacle CN-LG32 as seen by the technician approaching with the plug\n(i.e., looking into the socket contact face). Orientation reference:\nKey Position A at 12 o'clock. All pin positions are per MIL-DTL-38999 Series III\nInsert Arrangement 4-4.\n\n```\n        CONNECTOR CN-LG32 \u2014 SOCKET RECEPTACLE\n        MATING FACE VIEW (Technician Line-of-Sight to Socket)\n        MIL-DTL-38999 Series III \u2014 Insert Arrangement 4-4\n        Shell Size 11 \u2014 4 Contacts\n\n                KEY POSITION A\n                     \u2506  12 o'clock\n                     \u25bc\n              \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u25b2\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n           \u2571               \u2572\n          \u2502   \u250c\u2500\u2500\u2500\u2510         \u2502\n          \u2502   \u2502 A \u2502         \u2502  \u25c4\u2500\u2500 Pin A  (+28 VDC Power Input)\n          \u2502   \u2514\u2500\u2500\u2500\u2518         \u2502         9 o'clock position\n          \u2502                 \u2502\n    9h \u2500\u2500\u25ba\u2502  \u250c\u2500\u2500\u2500\u2510   \u250c\u2500\u2500\u2500\u2510  \u2502\u25c4\u2500\u2500 3h\n          \u2502  \u2502 B \u2502   \u2502 C \u2502  \u2502\n          \u2502  \u2514\u2500\u2500\u2500\u2518   \u2514\u2500\u2500\u2500\u2518  \u2502  Pin B (9 o'clock) \u2014 DC Ground Return\n          \u2502                 \u2502  Pin C (3 o'clock) \u2014 Analog Signal Output\n          \u2502   \u250c\u2500\u2500\u2500\u2510         \u2502\n          \u2502   \u2502 D \u2502         \u2502  \u25c4\u2500\u2500 Pin D  (Shield Drain \u2014 EMI Ground)\n          \u2502   \u2514\u2500\u2500\u2500\u2518         \u2502         6 o'clock position\n           \u2572               \u2571\n              \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n\n                  6 o'clock\n\n        \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n        \u2502  ORIENTATION LEGEND                     \u2502\n        \u2502  Key Position A : 12 o'clock (north)    \u2502\n        \u2502  Pin A          :  9 o'clock (west)     \u2502\n        \u2502  Pin B          :  9 o'clock lower      \u2502\n        \u2502  Pin C          :  3 o'clock (east)     \u2502\n        \u2502  Pin D          :  6 o'clock (south)    \u2502\n        \u2502                                         \u2502\n        \u2502  View Direction : INTO socket face      \u2502\n        \u2502  (Plug approaching from reader's side)  \u2502\n        \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n```\n\n> \u2139\ufe0f **NOTE:** The key position A anti-rotation feature is a physical bayonet\n> lug on the shell at 12 o'clock. Connector CN-LG32 is keyed to prevent\n> cross-mating with connectors CN-LG30 (Anti-skid control, key position B)\n> and CN-LG33 (Wheel Speed Transducer, key position D) located on the same\n> bracket BKT-32-LG-04. Verify key alignment visually before applying\n> mating torque.\n\n---\n\n### 6. PIN-OUT DISTRIBUTION TABLE \u2014 CN-LG32\n\n#### 6.1 Master Pin Assignment Summary\n\n| Pin | Wire Identifier      | AWG  | Core Count | Insulation Color | Functional Role Summary              | Destination                     |\n|-----|----------------------|------|------------|------------------|--------------------------------------|---------------------------------|\n| A   | W-32-MG102-A22       | 22   | 1 (single) | Red              | +28 VDC Power Input                  | CB-10MG, Panel 10VU             |\n| B   | W-32-MG103-B22       | 22   | 1 (single) | Black            | DC Ground Return to GD-02            | Ground Block GD-02, Frame 24    |\n| C   | W-32-MG104-C24       | 24   | 1 (paired) | Blue             | 1.0 V\u20135.0 V Analog Telemetry Signal  | BSCU-1, Rack 8VU, Input AV-12   |\n| D   | W-32-MG105-D-SH      | 24   | Shield     | Silver (bare)    | Outer Braid EMI Shield \u2014 GD-02       | Ground Block GD-02, Frame 24    |\n\n---\n\n#### 6.2 Detailed Pin-by-Pin Specification\n\n---\n\n##### PIN A \u2014 +28 VDC POWER INPUT\n\n| Field                        | Value                                                                       |\n|------------------------------|-----------------------------------------------------------------------------|\n| **Pin Identifier**           | Pin A                                                                       |\n| **Wire Identifier**          | W-32-MG102-A22                                                              |\n| **Core Gauge**               | 22 AWG (cross-section: 0.324 mm\u00b2 nominal per MIL-W-22759/16)               |\n| **Core Count**               | 1 (single conductor, unshielded on this segment)                           |\n| **Insulation Material**      | PTFE (Polytetrafluoroethylene), 0.25 mm wall, per MIL-W-22759/16           |\n| **Insulation Color**         | Red                                                                         |\n| **Voltage Rating**           | 600 VAC / 600 VDC continuous per MIL-W-22759/16                            |\n| **Temperature Rating**       | -65\u00b0C to +200\u00b0C                                                             |\n| **Functional Role**          | +28 VDC positive excitation supply to Brake Pressure Transducer FIN 12MG Wheatstone bridge Pin 1 (V+) |\n| **Signal Type**              | DC Power \u2014 non-switched, continuously energized when CB-10MG is closed     |\n| **Nominal Voltage**          | +28.0 VDC (tolerance: +27.5 VDC to +28.5 VDC under load)                  |\n| **Maximum Current**          | 0.25 A (fused at CB-10MG, 1 A, providing 4:1 over-current margin)          |\n| **Wire Run Origin**          | Circuit Breaker CB-10MG, Contact 1 (output), Panel 10VU, Row E, Position 14, Zone 120 |\n| **Wire Run Destination**     | CN-LG32 Pin A (Zone 710) \u2192 Transducer FIN 12MG internal bridge V+ terminal |\n| **Harness Bundle**           | W-32-MG100-SERIES, Sub-bundle W-32-MG102                                   |\n| **Conduit**                  | C-32-LG-01 from Frame 24 to Zone 710 bracket BKT-32-LG-04                 |\n| **Splice Points**            | SP-32-MG102-1 (Zone 310, Frame 44 lower spar, wire gauge maintained 22 AWG throughout) |\n| **Termination \u2014 Zone 710**   | Crimp socket contact, size 20, gold-plated, AMP P/N 336430-1 (22\u201324 AWG crimp range), seated in CN-LG32 insert position A |\n| **Termination \u2014 Zone 120**   | Ring lug, M4, tin-plated copper, crimped, terminated at CB-10MG output stud |\n| **Wire Length (approx.)**    | 12.4 m (40.7 ft) Zone 120 to Zone 710 routed length                        |\n| **Resistance (max)**         | 0.58 \u03a9 end-to-end (22 AWG PTFE, 12.4 m at 20\u00b0C)                           |\n| **Voltage Drop (max)**       | 0.145 V at 0.25 A maximum load (within +28 VDC \u00b10.5 V tolerance)          |\n| **Contact Extraction Tool**  | ABS-CON-EXT-38999, size 20 socket removal tip                               |\n| **Contact Insertion Tool**   | AMP P/N 91285-1, size 20 crimp and insertion tool set                      |\n\n---\n\n##### PIN B \u2014 DC GROUND RETURN\n\n| Field                        | Value                                                                       |\n|------------------------------|-----------------------------------------------------------------------------|\n| **Pin Identifier**           | Pin B                                                                       |\n| **Wire Identifier**          | W-32-MG103-B22                                                              |\n| **Core Gauge**               | 22 AWG (cross-section: 0.324 mm\u00b2 nominal per MIL-W-22759/16)               |\n| **Core Count**               | 1 (single conductor; forms twisted shielded pair with Pin C wire W-32-MG104-C24 from CN-LG32 rearward to Frame 24) |\n| **Insulation Material**      | PTFE, 0.25 mm wall, per MIL-W-22759/16                                     |\n| **Insulation Color**         | Black                                                                       |\n| **Voltage Rating**           | 600 VAC / 600 VDC continuous                                                |\n| **Temperature Rating**       | -65\u00b0C to +200\u00b0C                                                             |\n| **Functional Role**          | DC negative ground return path from Brake Pressure Transducer FIN 12MG bridge Pin 2 (V\u2212) to Ground Block GD-02, Frame 24 |\n| **Signal Type**              | DC Ground Return \u2014 low-impedance reference return, not signal-carrying       |\n| **Nominal Voltage**          | 0.0 VDC (ground reference)                                                  |\n| **Maximum Current**          | 0.25 A (matched to Pin A supply)                                            |\n| **Wire Run Origin**          | CN-LG32 Pin B (Zone 710) \u2190 Transducer FIN 12MG internal bridge V\u2212 terminal |\n| **Wire Run Destination**     | Ground Block GD-02, Lug Position 7, M4 stud, Frame 24, Zone 120, Fwd Bulkhead Lower Stringer |\n| **Harness Bundle**           | W-32-MG100-SERIES, twisted pair with W-32-MG104-C24 from CN-LG32 rearward  |\n| **Twist Rate**               | 12 \u00b1 2 twists per 300 mm (12 in) per AWM Chapter 20-54 Table 4             |\n| **Shielding**                | Enclosed within outer braid shield W-32-MG105-D-SH from CN-LG32 backshell to GD-02 |\n| **Splice Points**            | None \u2014 continuous run, no splices permitted on ground return per AWM Chapter 24-50 |\n| **Termination \u2014 Zone 710**   | Crimp socket contact, size 20, gold-plated, AMP P/N 336430-1, seated in CN-LG32 insert position B |\n| **Termination \u2014 Zone 120**   | Ring lug, M4, tin-plated copper, crimped to GD-02 Lug 7, torque 18 in-lbs (2.0 N\u00b7m) |\n| **Wire Length (approx.)**    | 12.4 m (40.7 ft), matched length to W-32-MG104-C24 within \u00b150 mm          |\n| **Resistance (max)**         | 0.58 \u03a9 end-to-end at 20\u00b0C. Ground block GD-02 to airframe structure: < 2.5 m\u03a9 per AWM Ch. 20-93 |\n| **Isolation \u2014 to Pin C**     | Minimum 10 M\u03a9 pin-to-pin isolation at 500 VDC Megger test (signal to ground, connector unmated) |\n| **Contact Extraction Tool**  | ABS-CON-EXT-38999, size 20 socket removal tip                               |\n| **Contact Insertion Tool**   | AMP P/N 91285-1, size 20 crimp and insertion tool set                      |\n\n---\n\n##### PIN C \u2014 ANALOG TELEMETRY SIGNAL OUTPUT\n\n| Field                        | Value                                                                       |\n|------------------------------|-----------------------------------------------------------------------------|\n| **Pin Identifier**           | Pin C                                                                       |\n| **Wire Identifier**          | W-32-MG104-C24                                                              |\n| **Core Gauge**               | 24 AWG (cross-section: 0.205 mm\u00b2 nominal per MIL-W-22759/16)               |\n| **Core Count**               | 1 (single conductor; forms twisted shielded pair with Pin B wire W-32-MG103-B22) |\n| **Insulation Material**      | PTFE, 0.20 mm wall, per MIL-W-22759/16                                     |\n| **Insulation Color**         | Blue                                                                        |\n| **Voltage Rating**           | 600 VAC / 600 VDC continuous                                                |\n| **Temperature Rating**       | -65\u00b0C to +200\u00b0C                                                             |\n| **Functional Role**          | 1.0 VDC to 5.0 VDC analog pressure telemetry signal output from Brake Pressure Transducer FIN 12MG bridge output Pin 3 (VOUT) to BSCU-1 analog voltage input channel AV-12, Main Avionics Rack 8VU |\n| **Signal Type**              | Analog, single-ended, DC-coupled, ratiometric output proportional to brake line hydraulic pressure |\n| **Signal Range**             | 1.0 VDC = 0 bar (0 PSI) \u2014 5.0 VDC = 350 bar (5,076 PSI)                   |\n| **Signal Linearity**         | \u00b1 0.5% full-scale (transducer specification, FIN 12MG bench test per AMM Task 32-42-05-400-001) |\n| **Out-of-Range Flags**       | < 0.8 VDC = open-circuit / low fault; > 5.2 VDC = short-to-power fault. Either condition sustained > 2.0 sec triggers CMS-FAULT-32-42-E12 at BSCU-1 |\n| **Source Impedance**         | 2.0 k\u03a9 nominal (transducer output bridge resistance, FIN 12MG)              |\n| **Input Impedance at BSCU-1**| 100 k\u03a9 (BSCU-1 analog input AV-12) \u2014 provides 50:1 load ratio, < 2% signal attenuation |\n| **Wire Run Origin**          | CN-LG32 Pin C (Zone 710) \u2190 Transducer FIN 12MG internal bridge VOUT terminal |\n| **Wire Run Destination**     | BSCU-1, Main Avionics Rack 8VU, Shelf 3, Position 2, Connector J8VU-32, Pin 12 (Analog Input AV-12) |\n| **Harness Bundle**           | W-32-MG100-SERIES, twisted pair with W-32-MG103-B22; enclosed in shield W-32-MG105-D-SH |\n| **Twist Rate**               | 12 \u00b1 2 twists per 300 mm (12 in) per AWM Chapter 20-54 Table 4             |\n| **Shielding**                | Outer braid W-32-MG105-D-SH \u2014 single-point ground at Frame 24 GD-02 only. Zone 710 end floating. |\n| **Splice Points**            | None \u2014 continuous run without splices mandatory per signal integrity requirements, AWM Chapter 20-54 Clause 6.3 |\n| **Termination \u2014 Zone 710**   | Crimp socket contact, size 20, gold-plated, AMP P/N 336430-1, seated in CN-LG32 insert position C |\n| **Termination \u2014 Zone 120**   | Crimp pin contact, size 20, gold-plated, seated in BSCU-1 mating connector J8VU-32 Pin 12 |\n| **Wire Length (approx.)**    | 12.4 m (40.7 ft), matched length to W-32-MG103-B22 within \u00b150 mm          |\n| **Capacitance (max)**        | 98 pF total (24 AWG PTFE, 12.4 m at 8 pF/m per MIL-W-22759/16) \u2014 well below BSCU-1 input bandwidth limit |\n| **Isolation \u2014 to Pin A**     | Minimum 10 M\u03a9 at 500 VDC Megger test (signal to +28 V power, connector unmated) |\n| **EMI Sensitivity**          | Susceptible to induced noise above 10 mV peak on this low-level signal. Do not route alongside high-current AC bus feeders. Maintain minimum 50 mm (2 in) physical separation from 115 VAC wiring bundles per AWM Chapter 20-54 Clause 9.1 |\n| **Contact Extraction Tool**  | ABS-CON-EXT-38999, size 20 socket removal tip                               |\n| **Contact Insertion Tool**   | AMP P/N 91285-1, size 20 crimp and insertion tool set                      |\n\n---\n\n##### PIN D \u2014 OUTER BRAID EMI SHIELD DRAIN \u2014 GROUND TERMINATION\n\n| Field                        | Value                                                                       |\n|------------------------------|-----------------------------------------------------------------------------|\n| **Pin Identifier**           | Pin D                                                                       |\n| **Wire Identifier**          | W-32-MG105-D-SH                                                             |\n| **Core Gauge**               | 24 AWG drain wire (bare tinned copper, extracted from braid at termination point) |\n| **Core Count**               | Shield \u2014 outer tinned copper braid, 90% optical coverage minimum            |\n| **Insulation Material**      | None (bare braid); drain wire has clear PTFE sleeve, 20 mm at each termination |\n| **Insulation Color**         | Silver (bare tinned copper braid); drain wire: clear/natural PTFE           |\n| **Voltage Rating**           | Ground reference \u2014 0 VDC nominal; withstands 50 V transient per DO-160G Section 22 |\n| **Temperature Rating**       | -65\u00b0C to +200\u00b0C                                                             |\n| **Functional Role**          | Outer braid EMI shield drain wire, grounded at Bulkhead Frame 24, Ground Block GD-02 Lug Position 8, to suppress radiated and conducted electromagnetic interference on analog telemetry signal wire W-32-MG104-C24 (Pin C) |\n| **Grounding Architecture**   | Single-point ground \u2014 Zone 710 end of shield braid is floating (unterminated, insulated with heat-shrink at CN-LG32 backshell). Frame 24 end only is grounded. Eliminates ground loop on low-level analog signal circuit. |\n| **Shield Coverage**          | Minimum 90% optical braid coverage per MIL-W-22759/16. Verify by inspection prior to harness installation. |\n| **Shield Continuity**        | Braid resistance Pin D to GD-02 Lug 8: < 0.5 \u03a9 measured end-to-end with DMM at 200 mA test current |\n| **Shield Isolation (Zone 710 end)** | Braid float resistance to airframe structure at Zone 710: > 1 M\u03a9 (verified Megger test, 500 VDC, connector mated) |\n| **Wire Run Origin**          | CN-LG32 Pin D (Zone 710, braid drain wire extracted at backshell entry point) |\n| **Wire Run Destination**     | Ground Block GD-02, Lug Position 8, M4 stud, Frame 24, Zone 120            |\n| **Harness Bundle**           | W-32-MG100-SERIES outer sheath \u2014 braid is the outermost layer enclosing twisted pair W-32-MG103-B22 / W-32-MG104-C24 |\n| **Splice Points**            | None permitted on shield drain wire per AWM Chapter 20-54 Clause 8.2       |\n| **Termination \u2014 Zone 710**   | Braid pigtail drain wire, 50 mm exposed, insulated with clear PTFE heat-shrink. Drain wire crimped to CN-LG32 Pin D socket contact, size 20, AMP P/N 336430-1. Backshell clamped over outer braid jacket entry \u2014 do not clamp over braid directly. |\n| **Termination \u2014 Zone 120**   | Drain wire, ring lug M4, tin-plated copper, crimped to GD-02 Lug 8, torque 18 in-lbs (2.0 N\u00b7m). Install bonding washer between lug and bus bar stud per AWM Chapter 20-93. |\n| **Contact Extraction Tool**  | ABS-CON-EXT-38999, size 20 socket removal tip                               |\n| **Contact Insertion Tool**   | AMP P/N 91285-1, size 20 crimp and insertion tool set                      |\n\n---\n\n### 7. CIRCUIT PROTECTION \u2014 CB-10MG\n\n| Field                        | Value                                                                       |\n|------------------------------|-----------------------------------------------------------------------------|\n| **Circuit Breaker Reference**| CB-10MG                                                                     |\n| **Panel Location**           | Panel 10VU, Row E, Position 14, Zone 120 (Main Equipment Centre)            |\n| **Rating**                   | 1 Ampere, 28 VDC, thermal trip                                              |\n| **Protected Circuit**        | W-32-MG102-A22 \u2014 Brake Pressure Transducer FIN 12MG excitation supply       |\n| **Trip Class**                | Class 2, slow-blow (sustained overload protection \u2014 not fast fault clearing) |\n| **Reset Type**               | Manual push-pull, non-latch. Collar color: Blue (ATA 32 system identification) |\n| **Placard**                  | `BRAKES / PRESS XDCR / LH` (white on black, 6-point font minimum)          |\n| **In-Flight Reset Authority**| Flight Crew \u2014 Single reset attempt permitted per FCOM Abnormal Procedure 32-42-01 |\n| **Ground Reset Authority**   | Maintenance personnel \u2014 Unlimited resets; investigate fault cause if CB trips repeatedly |\n| **Coordination with BSCU-1** | Loss of CB-10MG results in Pin A supply < 0.8 VDC at transducer, causing Pin C output to drop below 0.8 VDC. BSCU-1 detects low-signal fault and generates CMS-FAULT-32-42-E12 within 2.0 sec. |\n\n---\n\n### 8. WIRING CONTINUITY AND ISOLATION TEST PROCEDURE\n\n> \u2139\ufe0f **NOTE:** Perform the following tests with CB-10MG OPEN (pulled), aircraft\n> power OFF, and connectors CN-LG32 and J8VU-32 both unmated. Use calibrated\n> Digital Multimeter T11 (CAT III, 600 V) and Megger tester (500 VDC output)\n> for isolation tests.\n\n| Test | Pin(s) Tested          | Test Method                         | Acceptable Result                     | Reject If                         |\n|------|------------------------|-------------------------------------|---------------------------------------|-----------------------------------|\n| T01  | Pin A \u2014 continuity     | DMM resistance, Pin A to CB-10MG output stud | < 0.8 \u03a9 end-to-end          | \u2265 0.8 \u03a9 (open or high resistance)|\n| T02  | Pin B \u2014 continuity     | DMM resistance, Pin B to GD-02 Lug 7 | < 0.8 \u03a9 end-to-end                  | \u2265 0.8 \u03a9                           |\n| T03  | Pin C \u2014 continuity     | DMM resistance, Pin C to J8VU-32 Pin 12 | < 1.2 \u03a9 end-to-end             | \u2265 1.2 \u03a9                           |\n| T04  | Pin D \u2014 shield continuity | DMM resistance, Pin D to GD-02 Lug 8 | < 0.5 \u03a9 end-to-end             | \u2265 0.5 \u03a9                           |\n| T05  | Pin A to Pin B         | Megger 500 VDC, pin-to-pin isolation | \u2265 10 M\u03a9                              | < 10 M\u03a9 (insulation breakdown)   |\n| T06  | Pin A to Pin C         | Megger 500 VDC, pin-to-pin isolation | \u2265 10 M\u03a9                              | < 10 M\u03a9                           |\n| T07  | Pin A to Pin D         | Megger 500 VDC, pin-to-pin isolation | \u2265 10 M\u03a9                              | < 10 M\u03a9                           |\n| T08  | Pin B to Pin C         | Megger 500 VDC, pin-to-pin isolation | \u2265 10 M\u03a9                              | < 10 M\u03a9                           |\n| T09  | Pin D float (Zone 710) | Megger 500 VDC, Pin D braid to airframe, Zone 710 | \u2265 1 M\u03a9          | < 1 M\u03a9 (shield grounded at wrong end \u2014 ground loop fault) |\n| T10  | GD-02 ground bond      | DMM resistance, GD-02 Lug bus to Frame 24 structure | < 2.5 m\u03a9       | \u2265 2.5 m\u03a9 (degraded airframe bond)|\n\n---\n\n### 9. FAULT SIGNATURES AND WIRING CROSS-REFERENCE\n\n| Fault Code              | Signal Condition at BSCU-1 AV-12 | Probable Wiring Cause                      | Isolation Starting Point               |\n|-------------------------|----------------------------------|--------------------------------------------|----------------------------------------|\n| `CMS-FAULT-32-42-E12`   | Pin C < 0.8 VDC sustained        | Open circuit \u2014 Pin A (no excitation) or Pin C signal wire | Test T01 (Pin A continuity) first     |\n| `CMS-FAULT-32-42-E12`   | Pin C > 5.2 VDC sustained        | Pin C shorted to Pin A (+28 V contamination) | Test T06 (A-to-C isolation)           |\n| `CMS-FAULT-32-42-E13`   | Pin C noisy / oscillating        | Shield drain open \u2014 Pin D continuity loss or Zone 710 braid grounded creating ground loop | Test T04 and T09                       |\n| `CMS-FAULT-32-42-E14`   | Pin C fixed at 1.0 VDC (0 bar)  | Pin B ground return open \u2014 bridge reference floating | Test T02 (Pin B continuity)           |\n| ECAM `BRAKE L/R SYS FAULT` | Any above condition at BSCU-1 | Refer to `TSM DMC-A320-A-32-42-00-00A-520A-A` | Full fault isolation per TSM Step 3   |\n\n---\n\n### 10. REVISION RECORD\n\n| Rev | Date       | Reason for Change                                                     | Author     |\n|-----|------------|-----------------------------------------------------------------------|------------|\n| A   | 2022-06-01 | Initial issue                                                         | Avionics Eng. |\n| B   | 2023-03-14 | Connector P/N updated to D38999/26WA-4SN; backshell type added        | Avionics Eng. |\n| C   | 2023-11-08 | Fault signatures table added; CB-10MG trip class clarified            | Avionics Eng. |\n| D   | 2024-11-15 | Shield grounding architecture section expanded; Test T09 added        | Avionics Eng. |\n\n---\n\n*End of Data Module \u2014 DMC-A320-A-32-42-11-00A-520A-A*\n*AWM/WDM Wiring Architecture and Pin-Out Specification \u2014 Revision D \u2014 2024-11-15*\n```"}
for name, text in MANUALS.items():
    (APP_DIR / "manuals" / name).write_text(text, encoding="utf-8")
print("Demo application prepared at", APP_DIR)


In [ ]:
# Quick syntax check
!python -m py_compile /content/airbus_brake_streamlit_poc/app.py


In [ ]:
# Launch Streamlit in the background
%cd /content/airbus_brake_streamlit_poc
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &
print("Streamlit starting on port 8501...")


In [ ]:
# Create a public URL. Keep this cell running while you use the app.
print("If localtunnel asks for a password, use this IP:")
!curl -s https://ipv4.icanhazip.com
!npx --yes localtunnel --port 8501


## Suggested test questions

- What torque is required for the structural retention bolts, and what sequence should I follow?
- CMS-FAULT-32-42-E12 is shown. What is the first isolation step and possible causes?
- For connector CN-LG32 Pin C, give the signal role, wire identifier, and termination.
- Which actuator part number applies to MSN 001-200 and what O-ring kit is required?
- What pressure must be confirmed before loosening any brake line union?
